# Monthly Velocity Estimates for NISAR Cal/Val GPS Stations
---


Loops over NIT GPS stations, computes monthly regression velocities and a full-period regression for each, produces a summary plot, and saves results to YAML.


## Python Setup


In [ ]:
%load_ext autoreload
%autoreload 2
import time
#
reset = False
try:
    import nisarcryodb
except Exception:
    %pip install -e ~/./nisarcryodb
    reset = True
try:
    import nisargps
except Exception:
    %pip install -e ~/./nisargps
    reset = True
if reset:
    print('\n\033[1;31m\n\nRestart kernel and run this cell again \n\n \033[0m\n')
    time.sleep(1e9)


In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import nisargps
import nisarcryodb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from calendar import monthrange
import pandas as pd
import yaml


## Connect to Database


In [ ]:
configFile = '../nisarcryodb/Notebooks/calvaldb_config.ini'
DB = nisarcryodb.nisarcryodb(configFile=configFile)


## Compute Monthly and Full-Period Velocities

For each station: regression velocity for each complete month (skipping months with < 15 days of data) and regression over the full available period.


In [ ]:
%%time
stations = ['NIT0', 'NIT1', 'NIT2', 'NIT3', 'NIT5']
results = {}

for stationName in stations:
    print(f'\nProcessing {stationName}...')
    station = nisargps.nisarStation(stationName, DBConnection=DB, traceBack=False)
    if station.DBData is None or len(station.DBData) == 0:
        print(f'  No data found, skipping')
        continue

    times = pd.to_datetime(station.DBData['time_utc'])
    if times.dt.tz is not None:
        times = times.dt.tz_convert(None)
    t_min, t_max = times.min(), times.max()
    print(f'  Data range: {t_min.date()} to {t_max.date()}  ({len(station.DBData):,} points)')

    # Full-period regression
    vel_all = station.computeVelocity(
        t_min.strftime('%Y-%m-%d'), t_max.strftime('%Y-%m-%d'),
        method='regression', tides=False)
    if vel_all is None:
        print(f'  Full-period computeVelocity failed, skipping')
        continue
    vx_all, vy_all, _, _ = vel_all

    # Monthly regression — skip partial months (< 15 days of data)
    year_months = sorted(set(zip(times.dt.year.tolist(), times.dt.month.tolist())))
    month_dates, vx_monthly, vy_monthly = [], [], []
    for year, month in year_months:
        d1 = datetime(year, month, 1)
        d2 = datetime(year, month, monthrange(year, month)[1], 23, 59, 59)
        mask = (times >= pd.Timestamp(d1)) & (times <= pd.Timestamp(d2))
        if mask.sum() == 0:
            continue
        span_days = (times[mask].max() - times[mask].min()).total_seconds() / 86400
        if span_days < 15:
            continue
        vel = station.computeVelocity(d1, d2, method='regression', tides=False)
        if vel is None:
            continue
        vx, vy, _, _ = vel
        month_dates.append(d1)
        vx_monthly.append(float(vx))
        vy_monthly.append(float(vy))

    vx_arr = np.array(vx_monthly)
    vy_arr = np.array(vy_monthly)
    speed_all = float(np.sqrt(float(vx_all)**2 + float(vy_all)**2))

    results[stationName] = {
        'monthly': {
            'month': [d.strftime('%Y-%m') for d in month_dates],
            'vx': vx_monthly,
            'vy': vy_monthly
        },
        'total': {
            'vx': float(vx_all),
            'vy': float(vy_all),
            'speed': speed_all
        },
        'stats': {
            'vx_sigma': float(np.nanstd(vx_arr)),
            'vy_sigma': float(np.nanstd(vy_arr))
        }
    }
    print(f'  Full period: vx={vx_all:.2f}  vy={vy_all:.2f}  |v|={speed_all:.2f} m/yr')
    print(f'  Monthly std: sigma_vx={np.nanstd(vx_arr):.2f}  sigma_vy={np.nanstd(vy_arr):.2f} m/yr')
    print(f'  {len(month_dates)} complete months')


## Plot


In [ ]:
station_names = list(results.keys())
nStations = len(station_names)
fig, axes = plt.subplots(nStations, 2, figsize=(14, 4 * nStations), sharex=False)
if nStations == 1:
    axes = axes.reshape(1, -1)

for row, name in enumerate(station_names):
    data = results[name]
    month_dates = [datetime.strptime(m, '%Y-%m') for m in data['monthly']['month']]
    vx_arr = np.array(data['monthly']['vx'])
    vy_arr = np.array(data['monthly']['vy'])

    for col, (v_mon, v_all, sigma, label, color) in enumerate(zip(
            [vx_arr, vy_arr],
            [data['total']['vx'], data['total']['vy']],
            [data['stats']['vx_sigma'], data['stats']['vy_sigma']],
            ['$v_x$', '$v_y$'],
            ['r', 'b'])):
        ax = axes[row, col]
        if len(month_dates) > 0:
            ax.plot(month_dates, v_mon, 'o-', color=color, markersize=6,
                    linewidth=1.5,
                    label=f'{label} monthly ($\\sigma$={sigma:.2f} m/yr)')
            ax.axhline(v_all, color=color, linestyle='--', linewidth=1.5,
                       label=f'{label} full period = {v_all:.1f} m/yr')
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
            plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
        ax.set_ylabel(f'{label} (m/yr)', fontsize=12)
        ax.set_title(name, fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis='both', labelsize=10)

fig.suptitle('Monthly velocity estimates — NISAR Cal/Val GPS stations', fontsize=14)
plt.tight_layout()


## Save Results to YAML


In [ ]:
outFile = 'NIT_monthly_velocities.yaml'
with open(outFile, 'w') as f:
    yaml.dump(results, f, default_flow_style=False, sort_keys=False)
print(f'Saved to {outFile}')

# Summary
for name, data in results.items():
    n = len(data['monthly']['month'])
    vx, vy, spd = data['total']['vx'], data['total']['vy'], data['total']['speed']
    sx, sy = data['stats']['vx_sigma'], data['stats']['vy_sigma']
    print(f'{name}: {n} months  vx={vx:.2f}  vy={vy:.2f}  |v|={spd:.2f} m/yr  '
          f'sigma_vx={sx:.2f}  sigma_vy={sy:.2f}')
